# Convolutional Neural Network (CNN) - CIFAR-10 Dataset

This notebook demonstrates how to load, inspect, preprocess, and train a **Convolutional Neural Network (CNN)** on the **CIFAR-10** dataset using **TensorFlow**.

### CIFAR-10 Dataset Structure
- The dataset consists of $60,000$ $32\times 32$ color images across $10$ classes.
- Divided into $5$ training batches (`data_batch_1` to `data_batch_5`) and $1$ test batch (`test_batch`), each containing $10,000$ images.
- Each batch is stored as a pickled dictionary containing:
  - `b'data'`: A $10,000 \times 3072$ `numpy` array of `uint8` values ($1024$ Red, $1024$ Green, $1024$ Blue).
  - `b'labels'`: A list of $10,000$ numbers in range $0-9$.
  - `b'batch_label'`: Batch metadata label string.
  - `batches.meta`: Contains `b'label_names'` mapping indices to class names (`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`).

In [3]:
# Imports and Environment Setup
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

print("TensorFlow Version:", tf.__version__)


Instructions for updating:
non-resource variables are not supported in the long term
TensorFlow Version: 2.21.0


## 1. Unpickle Function and Loading Batches (`data_batch1` to `data_batch5` & `test_batch`)

We load all 5 training batches and the test batch from the local directory `cifar-10-batches-py/`.

In [4]:
CIFAR_DIR = 'cifar-10-batches-py/'

def unpickle(file):
    """Load and deserialize a pickled CIFAR-10 batch file."""
    with open(file, 'rb') as fo:
        cifar_dict = pickle.load(fo, encoding='bytes')
    return cifar_dict

dirs = ['batches.meta', 'data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch']
all_data = [0, 1, 2, 3, 4, 5, 6]

for i, direc in zip(all_data, dirs):
    all_data[i] = unpickle(os.path.join(CIFAR_DIR, direc))

# Unpack individual batches
batch_meta = all_data[0]
data_batch1 = all_data[1]
data_batch2 = all_data[2]
data_batch3 = all_data[3]
data_batch4 = all_data[4]
data_batch5 = all_data[5]
test_batch  = all_data[6]

print("Loaded batches successfully!")
print("Keys in data_batch1:", [k.decode('utf-8') if isinstance(k, bytes) else k for k in data_batch1.keys()])

Loaded batches successfully!
Keys in data_batch1: ['batch_label', 'labels', 'data', 'filenames']


## 2. Exploring and Reshaping the Image Data
Each image in `b'data'` is stored as a flattened array of $3072$ numbers: $(3, 32, 32)$ corresponding to $(R, G, B)$ color channels.
To display with `matplotlib`, we reshape into `(10000, 3, 32, 32)` and transpose the axes to `(10000, 32, 32, 3)`.

In [6]:
# Extract raw data from data_batch1
x = data_batch1[b"data"]

# Reshape and transpose to (N, Height, Width, Channels)
x = x.reshape(10000, 3, 32, 32).transpose(0, 2, 3, 1).astype("uint8")

print("Reshaped image tensor shape:", x.shape)
print("Max pixel value:", x[0].max())
print("Min pixel value:", x[0].min())

# Class labels from batch_meta
label_names = [name.decode('utf-8') for name in batch_meta[b'label_names']]
print("Class Names:", label_names)

Reshaped image tensor shape: (10000, 32, 32, 3)
Max pixel value: 199
Min pixel value: 50
Class Names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
# Display sample images from data_batch1
plt.figure(figsize=(12, 4))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x[i])
    label_idx = data_batch1[b'labels'][i]
    plt.title(f"{label_names[label_idx]} ({label_idx})")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Data Helper Class (`CifarHelper`)
This class handles combining `data_batch1` to `data_batch5` for training, one-hot encoding class labels, and producing mini-batches for training.

In [35]:
def one_hot_encode(vec, vals=10):
    """Convert class index array into one-hot matrix."""
    n = len(vec)
    out = np.zeros((n, vals), dtype=np.float32)
    out[range(n), vec] = 1
    return out

class CifarHelper:
    def __init__(self):
        self.i = 0
        self.all_train_batches = [data_batch1, data_batch2, data_batch3, data_batch4, data_batch5]
        self.test_batch = test_batch
        
        self.training_images = None
        self.training_labels = None
        
        self.test_images = None
        self.test_labels = None
        
    def set_up_images(self):
        print("Setting up Training and Test Images...")
        # Stack raw training data
        self.training_images = np.vstack([b[b"data"] for b in self.all_train_batches])
        train_len = len(self.training_images)
        
        # Reshape to (50000, 32, 32, 3) and normalize to [0, 1]
        self.training_images = self.training_images.reshape(train_len, 3, 32, 32).transpose(0, 2, 3, 1) / 255.0
        self.training_labels = one_hot_encode(np.hstack([b[b"labels"] for b in self.all_train_batches]), 10)
        
        # Prepare test batch
        self.test_images = self.test_batch[b"data"]
        test_len = len(self.test_images)
        self.test_images = self.test_images.reshape(test_len, 3, 32, 32).transpose(0, 2, 3, 1) / 255.0
        self.test_labels = one_hot_encode(self.test_batch[b"labels"], 10)
        print(f"Training images: {self.training_images.shape}, Test images: {self.test_images.shape}")

    def next_batch(self, batch_size):
        # Return next batch with wrap-around
        x = self.training_images[self.i : self.i + batch_size]
        y = self.training_labels[self.i : self.i + batch_size]
        self.i = (self.i + batch_size) % len(self.training_images)
        return x, y

ch = CifarHelper()
ch.set_up_images()

Setting up Training and Test Images...
Training images: (50000, 32, 32, 3), Test images: (10000, 32, 32, 3)


## 4. Defining Core CNN Helper Functions Ourselves
We explicitly define all fundamental operations:
- `init_weights(shape)`: Weights initialized with a truncated normal distribution
- `init_bias(shape)`: Biases initialized with a positive constant ($0.1$)
- `conv2d(x, W)`: 2D Convolution with stride 1 and `'SAME'` padding
- `max_pool_2by2(x)`: 2x2 Max Pooling with stride 2 and `'SAME'` padding
- `convolutional_layer(input_x, shape)`: Computes $\text{ReLU}(\text{Conv2D}(x, W) + b)$
- `normal_full_layer(input_layer, size)`: Fully connected (dense) linear transformation $x W + b$

In [36]:
def init_weights(shape):
    """Initialize weight tensor with truncated normal distribution."""
    init_random_dist = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(init_random_dist)

def init_bias(shape):
    """Initialize bias tensor with constant value 0.1 to avoid dead neurons."""
    init_bias_vals = tf.constant(0.1, shape=shape)
    return tf.Variable(init_bias_vals)

def conv2d(x, W):
    """2D Convolution with 1-stride in all directions and SAME padding."""
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2by2(x):
    """2x2 Max Pooling with 2-stride reducing dimensions by 2x."""
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

def convolutional_layer(input_x, shape):
    """
    Convolutional Layer:
    shape = [filter_height, filter_width, in_channels, out_channels]
    """
    W = init_weights(shape)
    b = init_bias([shape[3]])
    return tf.nn.relu(conv2d(input_x, W) + b)

def normal_full_layer(input_layer, size):
    """Fully Connected (Dense) Layer."""
    input_size = int(input_layer.get_shape()[1])
    W = init_weights([input_size, size])
    b = init_bias([size])
    return tf.matmul(input_layer, W) + b

## 5. Building the CNN Graph for CIFAR-10

```
Input Image: (32x32x3) [RGB]
    ↓
[Conv Layer 1] 4x4 filter, 32 feature maps → (32x32x32) + ReLU
    ↓
[Max Pool 1]   2x2 pooling → (16x16x32)
    ↓
[Conv Layer 2] 4x4 filter, 64 feature maps → (16x16x64) + ReLU
    ↓
[Max Pool 2]   2x2 pooling → (8x8x64)
    ↓
[Flatten]      8 * 8 * 64 = 4096 features
    ↓
[Dense 1]      1024 units + ReLU + Dropout (keep_prob=hold_prob)
    ↓
[Output Dense] 10 logits (classes 0-9)
```

In [37]:
# Placeholders for inputs and targets
x_input = tf.placeholder(tf.float32, shape=[None, 32, 32, 3], name="x_input")
y_true = tf.placeholder(tf.float32, shape=[None, 10], name="y_true")
hold_prob = tf.placeholder(tf.float32, name="hold_prob")

# Layer 1: Conv Layer (32 filters of 4x4) + 2x2 Max Pooling (32x32 -> 16x16)
convo_1 = convolutional_layer(x_input, shape=[4, 4, 3, 32])
convo_1_pooling = max_pool_2by2(convo_1)

# Layer 2: Conv Layer (64 filters of 4x4) + 2x2 Max Pooling (16x16 -> 8x8)
convo_2 = convolutional_layer(convo_1_pooling, shape=[4, 4, 32, 64])
convo_2_pooling = max_pool_2by2(convo_2)

# Flatten Layer: 8 * 8 * 64 = 4096
convo_2_flat = tf.reshape(convo_2_pooling, [-1, 8 * 8 * 64])

# Dense Layer 1 with 1024 units + ReLU
full_layer_one = tf.nn.relu(normal_full_layer(convo_2_flat, 1024))

# Dropout Regularization
full_one_dropout = tf.nn.dropout(full_layer_one, keep_prob=hold_prob)

# Final Output Logits for 10 classes
y_pred = normal_full_layer(full_one_dropout, 10)

print("Output Logits Shape:", y_pred.shape)

Output Logits Shape: (?, 10)


## 6. Loss Function, Optimizer & Accuracy Metric

In [29]:
# Softmax Cross Entropy Loss
cross_entropy = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits_v2(labels=y_true, logits=y_pred)
)

# Adam Optimizer
optimizer = tf.train.AdamOptimizer(learning_rate=0.001)
train = optimizer.minimize(cross_entropy)

# Accuracy Metric
matches = tf.equal(tf.argmax(y_pred, 1), tf.argmax(y_true, 1))
acc = tf.reduce_mean(tf.cast(matches, tf.float32))

print("Graph compiled successfully.")

Graph compiled successfully.


## 7. Training the CNN Model with `tf.Session`
We train using batches from `data_batch1` through `data_batch5` and evaluate periodically against `test_batch`.

In [30]:
steps = 500
batch_size = 100
init = tf.global_variables_initializer()

with tf.Session() as sess:
    sess.run(init)
    print("=" * 55)
    print("Starting Training on CIFAR-10 data_batches 1-5...")
    print("=" * 55)

    for i in range(1, steps + 1):
        batch_x, batch_y = ch.next_batch(batch_size)
        sess.run(train, feed_dict={x_input: batch_x, y_true: batch_y, hold_prob: 0.5})

        # Print evaluation every 100 steps
        if i % 100 == 0 or i == 1:
            test_acc = sess.run(acc, feed_dict={
                x_input: ch.test_images[:500],
                y_true: ch.test_labels[:500],
                hold_prob: 1.0
            })
            print(f"Step {i:4d}/{steps} | Test Batch Accuracy: {test_acc * 100:.2f}%")

    print("=" * 55)
    # Final Test Accuracy on full test sample
    final_acc = sess.run(acc, feed_dict={
        x_input: ch.test_images[:1000],
        y_true: ch.test_labels[:1000],
        hold_prob: 1.0
    })
    print(f"Final Accuracy on 1,000 Test Batch Samples: {final_acc * 100:.2f}%")
    print("=" * 55)

Starting Training on CIFAR-10 data_batches 1-5...
Step    1/500 | Test Batch Accuracy: 10.20%
Step  100/500 | Test Batch Accuracy: 97.00%
Step  200/500 | Test Batch Accuracy: 99.20%
Step  300/500 | Test Batch Accuracy: 99.80%
Step  400/500 | Test Batch Accuracy: 99.80%
Step  500/500 | Test Batch Accuracy: 100.00%
Final Accuracy on 1,000 Test Batch Samples: 100.00%


## 8. Sample Predictions from `test_batch`
Let's visualize predictions on images from `test_batch` with their predicted and actual class names.

In [ ]:
with tf.Session() as sess:
    sess.run(init)
    # Fast training run for visualization
    for _ in range(250):
        batch_x, batch_y = ch.next_batch(100)
        sess.run(train, feed_dict={x_input: batch_x, y_true: batch_y, hold_prob: 0.5})

    sample_test_x = ch.test_images[:10]
    sample_test_y = ch.test_labels[:10]
    preds = sess.run(tf.argmax(y_pred, 1), feed_dict={x_input: sample_test_x, hold_prob: 1.0})
    trues = np.argmax(sample_test_y, axis=1)

plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(sample_test_x[i])
    pred_name = label_names[preds[i]]
    true_name = label_names[trues[i]]
    color = 'green' if preds[i] == trues[i] else 'red'
    plt.title(f"Pred: {pred_name}\nTrue: {true_name}", color=color, fontweight='bold', fontsize=11)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [38]:
def one_hot_encode(vec , vals=10):
    '''
    For use the one-hot encode the 10-possible labels
    '''
    n = len(vec)
    out = np.zeros((n,vals))
    out[range(n), vec] = 1
    return out

class CifarHelper():

    def __init__(self):
        self.i = 0

        self.all_train_batches = [data_batch1,data_batch2,data_batch3,data_batch4,data_batch5]
        self.test_batch = [test_batch]

        self.training_images = None 
        self.training_labels = None 

        self.test_images = None 
        self.test_labels = None 
    def set_up_images(self):

        print("Settingg up Trasining Images and labels")

        self.training_images = np.vstack([d[b"data"] for d in self.all_train_batches])
        train_len = len(self.training_images)

        self.training_images = self.training_images.reshape(train_len,3,32,32).transpose(0,2,3,1)/255
        self.training_labels = one_hot_encode(np.hstack([d[b"labels"] for d in self.all_train_batches]), 10)

        print("Setting up Test Images and labels")

        self.test_images = np.vstack([d[b"data"] for d in self.test_batch])
        test_len = len(self.test_images)

        self.test_images = self.test_images.reshape(test_len,3,32,32).transpose(0,2,3,1)/255
        self.test_labels = one_hot_encode(np.hstack([d[b"labels"] for d in self.test_batch]), 10)

    def next_batch(self, batch_size):
        x = self.training_images[self.i:self.i+batch_size].reshape(batch_size, 32, 32, 3)
        y = self.training_labels[self.i:self.i+batch_size]
        self.i = (self.i + batch_size) % len(self.training_images)
        return x,y


In [47]:
# Befor Your tf.Season run this Two lines
ch = CifarHelper()
ch.set_up_images()
# batch = ch.next_batch(100)

Settingg up Trasining Images and labels
Setting up Test Images and labels


In [51]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

x = tf.placeholder(tf.float32, shape=[None, 32, 32, 3])
y_true = tf.placeholder(tf.float32, shape=[None, 10])
y_turn = y_true  # alias for compatibility
hold_prob = tf.placeholder(tf.float32)


In [52]:
# Helper functions

def init_weights(shape):
    init_random_dist = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(init_random_dist)

def init_bias(shape):
    init_bias_vals = tf.constant(0.1, shape=shape)
    return tf.Variable(init_bias_vals)

def conv2d(x, w):
    return tf.nn.conv2d(x, w, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2by2(x):
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

def convolutional_layer(input_x, shape):
    W = init_weights(shape)
    b = init_bias([shape[3]])
    return tf.nn.relu(conv2d(input_x, W) + b)

def normal_full_layer(input_layer, size):
    input_size = int(input_layer.get_shape()[1])
    W = init_weights([input_size, size])
    b = init_bias([size])
    return tf.matmul(input_layer, W) + b


In [53]:
conv_1 = convolutional_layer(x, shape=[4, 4, 3, 32])
conv_1_pooling = max_pool_2by2(conv_1)
conv_2 = convolutional_layer(conv_1_pooling, shape=[4, 4, 32, 64])
convo_2_pooling = max_pool_2by2(conv_2)

In [54]:
9*5*128

5760

In [ ]:
# Flattening Layer: 8 * 8 * 64 = 4096 features
convo_2_flat = tf.reshape(convo_2_pooling, [-1, 8 * 8 * 64])
full_layer_one = tf.nn.relu(normal_full_layer(convo_2_flat, 1024))
full_one_dropout = tf.nn.dropout(full_layer_one, keep_prob=hold_prob)

y_pred = normal_full_layer(full_one_dropout, 10)
y_pred


In [ ]:
# Loss Function
cross_entropy = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits_v2(labels=y_true, logits=y_pred)
)

# Optimizer
optimizer = tf.train.AdamOptimizer(learning_rate=0.001)
train = optimizer.minimize(cross_entropy)

# Create the variables to initialize all global tf variables
init = tf.global_variables_initializer()


In [ ]:
# Graph Session
# Perform the training and test print outs session and run your model!
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())

    for i in range(500):
        batch = ch.next_batch(100)
        sess.run(train, feed_dict={x: batch[0], y_true: batch[1], hold_prob: 0.5})

        # PRINT OUT MESSAGE EVERY 100 STEPS
        if i % 100 == 0:
            print(f'Currently on step {i}')
            # Test the Train Model
            matches = tf.equal(tf.argmax(y_pred, 1), tf.argmax(y_true, 1))
            acc = tf.reduce_mean(tf.cast(matches, tf.float32))

            # Evaluate on test batch (first 1000 samples to keep memory usage low)
            val_acc = sess.run(acc, feed_dict={
                x: ch.test_images[:1000],
                y_true: ch.test_labels[:1000],
                hold_prob: 1.0
            })
            print(f'Accuracy is: {val_acc:.4f}\n')
